# High-order polar splines

Many plasma-physics domains (cylinders, tori, stellarators) are parametrized by a **disk-like coordinate
transformation**

$$
\Phi(\hat x, \hat y) = \begin{bmatrix} R(\hat x, \hat y)\, C(\hat y) \\ R(\hat x, \hat y)\, S(\hat y)\end{bmatrix},
\qquad R(0, \hat y) = 0,
$$

which collapses the logical edge $\hat x = 0$ onto the centre (the *pole*, or the magnetic axis in 3D). Tensor-product
splines pushed forward by $\Phi$ are discontinuous at the pole and their gradients are unbounded. This tutorial shows
the **high-order polar subspaces** of R. Remmerswaal (talk *Constructing high-order de Rham complexes on 2D and 3D
domains with disk-like coordinate singularities via homotopy operators*, NMPP seminar 2026, and the Julia package
[PolarFEEC.jl](https://gitlab.mpcdf.mpg.de/rwr/polarfeec.jl)), which are available in Struphy as

```python
DerhamOptions(polar_splines=True, polar_method="homotopy", polar_regularity=r)
```

For a regularity $\mathfrak r$ the polar spaces are conforming ($\mathfrak r \geq 1$), the 0-forms are
$C^{\mathfrak r}$, the 1-forms $C^{\mathfrak r - 1}$ and the 2-forms $C^{\mathfrak r - 2}$ at the pole, they form a
de Rham complex with the cohomology of the disk and come with commuting projections:

$$
\bar V^{0,\mathfrak r} = \pi^0 T^{\mathfrak r} \oplus V^{0,\mathfrak r}_{\rm ext}, \qquad
\bar V^{1,\mathfrak r} = \operatorname{grad} \pi^0 T^{\mathfrak r} \oplus \operatorname{curl}^\dagger
    \tilde\pi^{2,1} T^{\mathfrak r - 2} \oplus V^{1,\mathfrak r}_{\rm ext}, \qquad
\bar V^{2,\mathfrak r} = \tilde\pi^{2,1} T^{\mathfrak r - 2} \oplus V^{2,\mathfrak r}_{\rm ext},
$$

where $T^{\mathfrak r}$ are the polynomials of total degree $\mathfrak r$, $\pi^k$ the commuting tensor-product
projections and $\operatorname{curl}^\dagger$ the Poincaré homotopy operator (a radial integral).

The computations of this tutorial use the serial core `struphy.polar.homotopy` (a port of PolarFEEC.jl); the last
section uses the same spaces in the 3D `Derham` complex. Set the environment variable `STRUPHY_TUTORIAL_FULL=1`
to run the studies at the resolution of the talk.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np

from struphy.polar.homotopy import (
    ClampedBSplineSpace,
    PeriodicBSplineSpace,
    PolarComplex2D,
    TrigonometricSpace,
    perturbed_disk,
)
from struphy.polar.homotopy.demos import basis, regularity

FULL = os.environ.get("STRUPHY_TUTORIAL_FULL", "0") == "1"

## 1. Tensor-product splines on a disk-like mapping

We use the perturbed disk of the talk, $R = \hat x\,(1 + 0.2\,\hat x \cos(6\pi \hat y))$, $C = \cos(2\pi\hat y)$,
$S = \sin(2\pi\hat y)$. Its Jacobian $J = R\,\partial_{\hat x}R\,(C S' - S C')$ vanishes only at $\hat x = 0$.

In [ ]:
basis.figure_mapping();

The tensor-product 0-forms are products $B_i(\hat x) U_j(\hat y)$ of cubic clamped B-splines on $N^x = 5$ cells and
cubic periodic B-splines on $N^y = 12$ cells (or trigonometric functions). The first ring ($i = 0$) collapses to the
pole:

In [ ]:
basis.figure_tensor_basis("bspline");

In [ ]:
basis.figure_tensor_basis("trig");

A generic tensor-product function is discontinuous at the origin, since $P(0) = \sum_j p_{0j} U_j(\hat y)$ depends on
the angle; its gradient is unbounded. The classical remedy is $p_{0j} = P(0)$ ($C^0$) and first-order conditions for
$C^1$ (Toshniwal et al. 2017; Holderied & Possanner 2022, the `polar_method="c1"` splines of Struphy).

In [ ]:
basis.figure_discontinuity();

## 2. High-order polar 0-forms

The polar subspace of regularity $\mathfrak r$ keeps the *external* tensor-product functions (rings $i > \mathfrak r$,
which vanish to order $\mathfrak r + 1$ at the pole) and adds back the $(\mathfrak r + 1)(\mathfrak r + 2)/2$
polynomials of degree $\mathfrak r$, projected onto the tensor-product space and truncated to the inner rings:

In [ ]:
for r in (1, 2, 3):
    basis.figure_polar_zero_forms(r)

The dimensions and the cohomology $(1, 0, 0)$ of the disk, for every regularity:

In [ ]:
radial, angular = ClampedBSplineSpace(3, 5), PeriodicBSplineSpace(3, 12)
for r in (0, 1, 2, 3):
    c = PolarComplex2D(radial, angular, perturbed_disk(), regularity=r)
    print(f"r = {r}: dimensions {[c.dimension(k) for k in range(3)]}, central functions {c.central_counts()}, "
          f"cohomology {c.cohomology_dimensions()}")

### $\mathfrak r$-compatibility

The polynomials are reproduced exactly (and the spaces are exactly $C^{\mathfrak r}$) when the pullback of
$T^{\mathfrak r}$ lies in the logical tensor-product space. For $R = R_1\hat x(1 + \varepsilon\hat x\cos(2\pi k\hat y))$
this needs clamped B-splines of degree $\geq 2\mathfrak r$ and the trigonometric modes up to $(k + 1)\mathfrak r$ in
$\hat y$. With B-splines in $\hat y$ the central functions are spline approximations of the polynomials and the
regularity holds up to that approximation error.

### Regularity at the centre (PolarFEEC.jl example 04)

A function is $C^k$ at the centre exactly when, for each $j \leq k$, its $j$-th derivative along the ray at angle
$\alpha$ contains only the angular harmonics $j, j-2, \dots$. We project
$f(x, y) = \sin(3x + 1)\cos(2y - 0.5) + 0.5\cos(4xy)$ with the commuting projections into the tensor-product space and
the polar spaces $\mathfrak r = 0, \dots, 3$ (degree-5 B-splines, 24 Fourier modes, a star-shaped disk) and measure the
remaining harmonics $\delta_k$ by polynomial fits along 48 rays:

In [ ]:
results = regularity.study()

In [ ]:
regularity.figure_central_basis(3, results);

In [ ]:
regularity.figure_regularity(results);

With a B-spline angular factor the regularity holds only up to the spline approximation of the polynomials:

In [ ]:
print("r = 3, B-spline angular factor:", regularity.bspline_angular_defects())

## 3. Polar 1- and 2-forms

Gradients of polynomials alone do not give a rich enough 1-form space, since the 2-form space contains no
polynomials. The paper enriches the 2-forms with projected polynomials $\tilde\pi^{2,1} T^{\mathfrak r - 2}$ and adds
their images under the homotopy operator $\operatorname{curl}^\dagger$ to the 1-forms. The resulting spaces form a
cochain complex (with $\operatorname{grad}$, $\operatorname{curl}$) and a chain complex (with the homotopies), with
$\bar V^{1,\mathfrak r} = \operatorname{grad} \bar V^{0,\mathfrak r} \oplus \operatorname{curl}^\dagger
\bar V^{2,\mathfrak r}$. The central basis functions for $\mathfrak r = 3$:

In [ ]:
basis.figure_all_forms(3);

## 4. Polar splines in the 3D Struphy complex

In 3D the polar complex is the total complex of the 2D polar complex of the cross-section and the 1D complex in
$\eta_3$. In Struphy it is selected with the `DerhamOptions` of the polar splines; the new domains
`PerturbedDiskCylinder` and `RoundedSquareCylinder` (and all domains with a pole, e.g. `HollowCylinder(a1=0)`,
`HollowTorus(a1=0)`, `IGAPolarTorus`) can be used:

In [ ]:
from struphy.feec.psydac_derham import Derham
from struphy.geometry import domains
from struphy.io.options import DerhamOptions
from struphy.topology.grids import TensorProductGrid

domain = domains.PerturbedDiskCylinder(R1=1.0, eps=0.2, k=3, Lz=1.0)
options = DerhamOptions(
    degree=(3, 3, 1),
    bcs=(("free", "free"), None, None),
    polar_splines=True,
    polar_method="homotopy",
    polar_regularity=3,
    nquads_proj=(8, 10, 4),
)
derham = Derham(TensorProductGrid(num_elements=(8, 12, 1)), options, domain=domain)
for space_id in ("H1", "Hcurl", "Hdiv", "L2"):
    space = derham.polar_coeff_spaces[derham.space_to_form[space_id]]
    print(f"{space_id:5s}: polar coefficients per component {space.n_polar}, polar rings {space.n_rings}")

The commuting projectors are the paper's $\bar\pi = \bar\Pi \circ \pi$: the tensor-product projection followed by the
polar dual coefficients. They commute with the polar derivatives (here for logical components, up to the finite
differences used to differentiate the test functions):

In [ ]:
h = 1e-6


def f(a, b, c):
    return np.sin(2 * np.pi * b) * a**2 + a**3 * np.cos(4 * np.pi * b) + 0 * c


def d(fun, i):
    def g(a, b, c):
        ep, em = [a, b, c], [a, b, c]
        ep[i] = ep[i] + h
        em[i] = em[i] - h
        return (fun(*ep) - fun(*em)) / (2 * h)

    return g


u = derham.P0(f)
g = derham.P1([d(f, 0), d(f, 1), d(f, 2)])
r = derham.grad.dot(u) - g
print("|| grad P0 f - P1 grad f || / || P1 grad f || =", np.sqrt(r.inner(r) / g.inner(g)))

## 5. Wave equations and guiding-centre orbits

The talk compares the regularities $\mathfrak r = 0, \dots, 5$ for the 0- and 1-form wave equations in 2D and 3D and
for guiding-centre orbits near the magnetic axis. These studies are run as Struphy examples (see
`examples/` and the documentation of the polar splines).